# Intent v2 Dataset QA Export

Checks the custom 3-label intent dataset before training. This notebook uses the standard-split story: `intent_train_v2.csv` has 1,680 balanced rows, then training notebook performs stratified 80/20 split.

In [ ]:
import os, sys, json
from pathlib import Path
import pandas as pd

if not Path("configs/config.yaml").exists():
    os.chdir("../..")
sys.path.insert(0, ".")

DATA_PATH = Path("data/en/intent_v2/intent_train_v2.csv")
OUT_DIR = Path("reports/en/intent_v2_standard_split")
OUT_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA_PATH)
print(df.shape)
print(df.head())
print(df["label"].value_counts())

In [ ]:
# Label balance check
counts = df["label"].value_counts().sort_index()
print(counts)
print("Balanced:", counts.nunique() == 1)

In [ ]:
# Exact duplicate check
exact_dupes = int(df.duplicated("text").sum())
print("Exact duplicate questions:", exact_dupes)

In [ ]:
# Semantic separation check with MiniLM embeddings
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
emb = model.encode(df["text"].tolist(), normalize_embeddings=True, show_progress_bar=True)
labels = df["label"].tolist()
sim = cosine_similarity(emb)
within, between = [], []
for i in range(len(df)):
    for j in range(i + 1, len(df)):
        if labels[i] == labels[j]:
            within.append(sim[i, j])
        else:
            between.append(sim[i, j])
within_avg = float(np.mean(within))
between_avg = float(np.mean(between))
ratio = within_avg / between_avg if between_avg > 0 else float("inf")
print(f"Avg within-class similarity:  {within_avg:.3f}")
print(f"Avg between-class similarity: {between_avg:.3f}")
print(f"Separation ratio: {ratio:.2f}x")

In [ ]:
# Export summary used by slides/report
summary = {
    "dataset_path": str(DATA_PATH),
    "rows": int(len(df)),
    "label_counts": {k: int(v) for k, v in counts.items()},
    "exact_duplicates": exact_dupes,
    "semantic_separation": {
        "within_avg": within_avg,
        "between_avg": between_avg,
        "ratio": ratio,
    },
    "recommended_training_split": "stratified 80/20",
    "expected_train_rows": int(round(len(df) * 0.8)),
    "expected_test_rows": int(len(df) - round(len(df) * 0.8)),
}
path = OUT_DIR / "dataset_qa_export_summary.json"
path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(summary, indent=2, ensure_ascii=False))
print("Saved:", path)